This notebook showcases the use of TxConformal in protein stability selection.

In [1]:
import os
import numpy as np 
import pickle
import pandas as pd  
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

from txconformal import TxConformal
from txconformal.features.providers import FeaturesProvider

In [6]:
seed = 0 
path = '/Users/yjinstat/Desktop/Research/collaborations/cf_drug_discovery/data/protein/'   
data = pd.read_csv(path + 'stability' + "_pred.csv") 
with open(path + 'stability' + "_test_emb.pkl", 'rb') as file:
    test_feature = pickle.load(file)
with open(path + 'stability' + "_calib_emb.pkl", 'rb') as file:
    calib_feature = pickle.load(file)  

data_cal = data[data['split'] == 'calib'].copy(deep=True)[['label', 'pred']].reset_index(drop=True)
data_test = data[data['split'] == 'test'].copy(deep=True)[['label', 'pred']].reset_index(drop=True)  

FDR_target = 0.6

fdp_list = []
for seed in range(10):
    res_cal, _, emb_cal, _ = train_test_split(data_cal, calib_feature, test_size=0.25, random_state=seed)
    res_test, _, emb_test, _ = train_test_split(data_test, test_feature, test_size=0.25, random_state=seed)
    y_calib = res_cal['label'].values
    f_calib = res_cal['pred'].values
    f_test = res_test['pred'].values

    c = np.quantile(data['label'], 0.8)

    prov = FeaturesProvider(f_calib=f_calib, f_test=f_test, E_calib=emb_cal, E_test=emb_test)
    prov.prepare()
    txc = TxConformal(score_name="clip",  
                    grid = [(x,y) for x in (0.1, 0.2, 0.5, 1) for y in (0.00001, 0.0001, 0.001, 0.005, 0.01, 0.02, 0.05, 0.1)],
                    tolerances=(1e-4, 1e-3, 1e-2))
    txc.fit(prov, y_calib, cutoff = c, print_level=-1)
    sel_res = txc.select(method="bh", alpha=FDR_target)
    sel_idx = sel_res.idx
    fdp_est = np.sum(res_test.iloc[sel_idx]['label']<= c) / len(sel_idx)  if len(sel_idx) > 0 else 0.0
    print("seed:", seed, "| selected:", len(sel_idx), "| estimated FDP:", fdp_est)
    fdp_list.append(fdp_est)

print("average estimated FDP:", np.mean(fdp_list))

seed: 0 | selected: 5518 | estimated FDP: 0.6404494382022472
seed: 1 | selected: 5891 | estimated FDP: 0.6596503140383636
seed: 2 | selected: 5841 | estimated FDP: 0.6630713918849512
seed: 3 | selected: 5875 | estimated FDP: 0.659063829787234
seed: 4 | selected: 5880 | estimated FDP: 0.6612244897959184
seed: 5 | selected: 5880 | estimated FDP: 0.6649659863945578
seed: 6 | selected: 5952 | estimated FDP: 0.6653225806451613
seed: 7 | selected: 6459 | estimated FDP: 0.6846261031119368
seed: 8 | selected: 5989 | estimated FDP: 0.6625480046752379
seed: 9 | selected: 5931 | estimated FDP: 0.6589108076209745
average estimated FDP: 0.6619832946156583
